# CVP mid-term, round 2: the SHARP CONTROL run (~45 min in total)

Colab → Runtime → Change runtime type → **T4 GPU**. Run cells **1–5, then 8–9**. Cells 6–7 are optional extras for later.

The sharp control trains on the same 30 views but with the *sharp* reference images. Comparing it with round 1 (strong motion, strong defocus) shows how much blur damages each structure type, versus what is hard to rebuild anyway.

Times on a free T4: install 5–10 min · data ~3 min · training ~26 min · scoring ~8–10 min.
On Colab, cell 2 asks for Google Drive access: the finished run is copied to `MyDrive/cvp_runs`.

In [ ]:
# 1) GPU check
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
# 2) Get the latest code + a safe place for results
import os
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
os.chdir(WORK)
if not os.path.exists("cvp_project"):
    !git clone -q https://github.com/Saksham-Bali/cvp_project
os.chdir(f"{WORK}/cvp_project")
!git pull -q
# Colab: keep finished runs on Google Drive, so a disconnect does not lose them (a window asks you to allow access)
if ON_KAGGLE:
    BACKUP = "/kaggle/working/cvp_runs_backup"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = "/content/drive/MyDrive/cvp_runs"
os.makedirs(BACKUP, exist_ok=True)
print("finished runs are copied to", BACKUP)
!ls scripts

In [ ]:
# 3) Install + compile gsplat (5-10 min the first time in a session)
!pip -q install gsplat==1.5.3 ninja jaxtyping rich plyfile huggingface_hub requests lpips
!python scripts/gsplat_smoke.py

In [ ]:
# 4) Data: strong-motion folder (it also holds the sharp reference images) + the laser scan
SCENE = "Ujikintoki"
!python scripts/download_realx3d.py --what quarter gt --scenes $SCENE --conditions motion_strong

In [ ]:
# 5) THE run for the mid-term: sharp control, 7000 steps (same settings as round 1), then scored.
!python scripts/run_batch.py --scene $SCENE --backup $BACKUP --runs motion_strong:sharp:7000:sharp_control

In [ ]:
# 6) OPTIONAL, later (not needed for the mid-term): mild blur, ~35-40 min each
# !python scripts/download_realx3d.py --what quarter --scenes $SCENE --conditions motion_mild defocus_mild
# !python scripts/run_batch.py --scene $SCENE --backup $BACKUP --runs motion_mild:blurred:7000 defocus_mild:blurred:7000

In [ ]:
# 7) OPTIONAL, later: strong motion with 30000 steps (~2.5 h)
# !python scripts/run_batch.py --scene $SCENE --backup $BACKUP --runs motion_strong:blurred:30000:vanilla_30k

In [ ]:
# 8) Table + figures of every finished run (reads BACKUP, so it includes runs from before a disconnect)
!python scripts/summarize_runs.py --scene $SCENE --runs $BACKUP
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"outputs/{SCENE}/ours/*.png")) + sorted(glob.glob(f"{BACKUP}/{SCENE}/*/*/s0/previews/*0001*.jpg")):
    print(p); display(Image(p, width=1100))

In [ ]:
# 9) Download the results (small files only: metrics, geometry, previews, figures).
#    Inside the zip the runs are under runs/ - extract it into D:\\cvp_project next to the round-1 runs.
import zipfile
with zipfile.ZipFile("cvp_results_round2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files_ in os.walk(BACKUP):
        for f in files_:
            full = os.path.join(root, f)
            z.write(full, os.path.join("runs", os.path.relpath(full, BACKUP)))
    for root, _, files_ in os.walk("outputs"):
        if "gt_cache" in root:
            continue
        for f in files_:
            if not f.endswith((".ply", ".npz")):
                z.write(os.path.join(root, f))
print(round(os.path.getsize("cvp_results_round2.zip") / 1e6, 1), "MB")
try:
    from google.colab import files; files.download("cvp_results_round2.zip")
except Exception:
    print("On Kaggle: download cvp_results_round2.zip from the Output panel (/kaggle/working/cvp_project)")